# HIPE-2026 Dataset Preparation Pipeline

**Objective:**
This notebook processes and prepares the multilingual HIPE-2026 dataset for relation extraction tasks. Specifically, it parses raw JSONL data in English, German, and French, extracts relevant entity pairs, and prepares balanced datasets for two specific relations: `'at'` and `'isAt'`. Finally, it performs an 80/20 train/evaluation split while maintaining class balance and exports the final sets for model training.

---
## 0. Setup and Environment
Mounting Google Drive to access the raw dataset files.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Data Loading and Preprocessing
In this section, we load the raw `.jsonl` files for the three target languages. We extract the textual context and entity pairs, retaining the language metadata for each record. The flattened structure is then loaded into a Pandas DataFrame for easier manipulation.

In [2]:
import json
import random
import pandas as pd

# Set paths
base_path = "/content/drive/MyDrive/colab_data/HIPE-2026-data/data/sandbox/"
input_files = ["en-train-cleaned.jsonl", "de-train-cleaned.jsonl", "fr-train-cleaned.jsonl"]

# 1. Load and flatten the data
all_pairs = []
for file_name in input_files:
    file_path = base_path + file_name
    lang = file_name.split('-')[0] # Extract language prefix (en, de, fr)
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                record = json.loads(line)
                text_context = record.get('text', '')

                # The pairs seem to be under 'sampled_pairs'
                if 'sampled_pairs' in record:
                    for pair in record['sampled_pairs']:
                        pair_data = pair.copy()
                        pair_data['text'] = text_context # Attach the text context to the pair
                        pair_data['lang'] = lang # Keep track of the language
                        all_pairs.append(pair_data)
                elif 'pairs' in record:
                    for pair in record['pairs']:
                        pair_data = pair.copy()
                        pair_data['text'] = text_context
                        pair_data['lang'] = lang # Keep track of the language
                        all_pairs.append(pair_data)
    except FileNotFoundError:
        print(f"File not found: {file_path}. Please check the path.")

df = pd.DataFrame(all_pairs)
print(f"Total entity pairs loaded across all languages: {len(df)}")
if not df.empty:
    display(df.head(2))

Total entity pairs loaded across all languages: 6170


,pers_entity_id,pers_wikidata_QID,pers_mentions_list,loc_entity_id,loc_wikidata_QID,loc_mentions_list,at,at_explanation,isAt,isAt_explanation,text,lang
0,sn83026170-1820-01-10-a-i0001-NIL_mr_pain_wareing,None,[Mr. Pain Wareing],sn83026170-1820-01-10-a-i0001_Q1333499,Q1333499,[Rappahannock],PROBABLE,,FALSE,,Was Committed to the Jail for the county of Al...,en
1,sn83026170-1820-01-10-a-i0001-NIL_mr_pain_wareing,None,[Mr. Pain Wareing],sn83026170-1820-01-10-a-i0001_Q88,Q88,[Alexandria],FALSE,,FALSE,,Was Committed to the Jail for the county of Al...,en


## 2. Dataset Preparation: `'at'` Relation
The first relation we focus on is `'at'`, which contains three valid labels: `TRUE`, `FALSE`, and `PROBABLE`.

To ensure our model doesn't become biased towards the majority class, we perform **stratified under-sampling** to balance the classes. We also shuffle the dataset to randomize the languages.

In [ ]:
# 2. Prepare Dataset A ('at' relation)
# Extract relevant columns and rename them to standard names (keep 'lang')
df_at = df[['text', 'pers_mentions_list', 'loc_mentions_list', 'at', 'lang']].copy()
df_at = df_at.rename(columns={'pers_mentions_list': 'person', 'loc_mentions_list': 'place', 'at': 'label'})

# Keep only valid labels for 'at' and balance the dataset
df_at = df_at[df_at['label'].isin(['TRUE', 'FALSE', 'PROBABLE'])]

# Balance (stratified under-sampling)
min_count_at = df_at['label'].value_counts().min()
print(f"\n'at' class distribution BEFORE balancing:\n{df_at['label'].value_counts()}")

df_at_balanced = df_at.groupby('label').sample(n=min_count_at, random_state=42)
print(f"\n'at' class distribution AFTER balancing:\n{df_at_balanced['label'].value_counts()}")

# Shuffle to completely randomize languages
df_at_balanced = df_at_balanced.sample(frac=1, random_state=42).reset_index(drop=True)


'at' class distribution BEFORE balancing:
label
FALSE       3669
PROBABLE    1579
TRUE         922
Name: count, dtype: int64

'at' class distribution AFTER balancing:
label
FALSE       922
PROBABLE    922
TRUE        922
Name: count, dtype: int64


## 3. Dataset Preparation: `'isAt'` Relation
The second relation is `'isAt'`, which contains two valid labels: `TRUE` and `FALSE`.

Similar to the previous step, we filter out invalid labels, balance the classes perfectly (1:1 ratio) using under-sampling, and shuffle the resulting dataset.

In [ ]:
# 3. Prepare Dataset B ('isAt' relation)
# Extract relevant columns and rename them to standard names (keep 'lang')
df_isAt = df[['text', 'pers_mentions_list', 'loc_mentions_list', 'isAt', 'lang']].copy()
df_isAt = df_isAt.rename(columns={'pers_mentions_list': 'person', 'loc_mentions_list': 'place', 'isAt': 'label'})

# Keep only valid labels for 'isAt' and balance the dataset
df_isAt = df_isAt[df_isAt['label'].isin(['TRUE', 'FALSE'])]

# Balance (1:1)
min_count_isAt = df_isAt['label'].value_counts().min()
print(f"\n'isAt' class distribution BEFORE balancing:\n{df_isAt['label'].value_counts()}")

df_isAt_balanced = df_isAt.groupby('label').sample(n=min_count_isAt, random_state=42)
print(f"\n'isAt' class distribution AFTER balancing:\n{df_isAt_balanced['label'].value_counts()}")

# Shuffle to completely randomize languages
df_isAt_balanced = df_isAt_balanced.sample(frac=1, random_state=42).reset_index(drop=True)


'isAt' class distribution BEFORE balancing:
label
FALSE    5493
TRUE      677
Name: count, dtype: int64

'isAt' class distribution AFTER balancing:
label
FALSE    677
TRUE     677
Name: count, dtype: int64


## 4. Train/Evaluation Split and Export
Finally, we split both the `'at'` and `'isAt'` datasets into training (80%) and evaluation (20%) sets. We use a stratified split approach to guarantee that the class distributions remain balanced in both the training and evaluation sets.

The final datasets are shuffled once more and exported to Google Drive as standard JSON files.

In [ ]:
import os

# 4. Split 80/20 and Save to Google Drive
os.makedirs(base_path, exist_ok=True)

# Split 'at' dataset (80% train, 20% eval) using pandas to maintain class balance
at_train = df_at_balanced.groupby('label').sample(frac=0.8, random_state=42)
at_eval = df_at_balanced.drop(at_train.index)

# Split 'isAt' dataset (80% train, 20% eval)
isAt_train = df_isAt_balanced.groupby('label').sample(frac=0.8, random_state=42)
isAt_eval = df_isAt_balanced.drop(isAt_train.index)

# Shuffle the resulting splits to ensure languages are mixed
at_train = at_train.sample(frac=1, random_state=42).reset_index(drop=True)
at_eval = at_eval.sample(frac=1, random_state=42).reset_index(drop=True)
isAt_train = isAt_train.sample(frac=1, random_state=42).reset_index(drop=True)
isAt_eval = isAt_eval.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"'at' Dataset -> Train size: {len(at_train)}, Eval size: {len(at_eval)}")
print(f"'isAt' Dataset -> Train size: {len(isAt_train)}, Eval size: {len(isAt_eval)}")

print("\n--- Language Distribution in Train Sets ---")
print(f"'at_train' languages:\n{at_train['lang'].value_counts()}")
print(f"\n'isAt_train' languages:\n{isAt_train['lang'].value_counts()}")
print("-------------------------------------------\n")

# Define output paths
paths = {
    'at_train': base_path + "at_train.json",
    'at_eval': base_path + "at_eval.json",
    'isAt_train': base_path + "isAt_train.json",
    'isAt_eval': base_path + "isAt_eval.json"
}

# Export datasets (dropping the 'lang' column to match previous format if desired, or keeping it. Let's keep it for now).
with open(paths['at_train'], 'w', encoding='utf-8') as f:
    json.dump(at_train.to_dict(orient='records'), f, indent=2, ensure_ascii=False)
with open(paths['at_eval'], 'w', encoding='utf-8') as f:
    json.dump(at_eval.to_dict(orient='records'), f, indent=2, ensure_ascii=False)
with open(paths['isAt_train'], 'w', encoding='utf-8') as f:
    json.dump(isAt_train.to_dict(orient='records'), f, indent=2, ensure_ascii=False)
with open(paths['isAt_eval'], 'w', encoding='utf-8') as f:
    json.dump(isAt_eval.to_dict(orient='records'), f, indent=2, ensure_ascii=False)

print("Successfully saved all splits:")
for name, path in paths.items():
    print(f" - {path}")

'at' Dataset -> Train size: 2214, Eval size: 552
'isAt' Dataset -> Train size: 1084, Eval size: 270

Successfully saved all splits:
 - /content/drive/MyDrive/colab_data/HIPE-2026-data/data/sandbox/at_train.json
 - /content/drive/MyDrive/colab_data/HIPE-2026-data/data/sandbox/at_eval.json
 - /content/drive/MyDrive/colab_data/HIPE-2026-data/data/sandbox/isAt_train.json
 - /content/drive/MyDrive/colab_data/HIPE-2026-data/data/sandbox/isAt_eval.json


## 5. Class Distribution per Language
Here we analyze the original distribution of the `at` and `isAt` labels for each language before balancing.

In [3]:
# Calculate distribution for 'at' relation per language
print("Class distribution for 'at' relation per language:")
display(df.groupby('lang')['at'].value_counts().unstack().fillna(0).astype(int))

print("\n" + "="*50 + "\n")

# Calculate distribution for 'isAt' relation per language
print("Class distribution for 'isAt' relation per language:")
display(df.groupby('lang')['isAt'].value_counts().unstack().fillna(0).astype(int))

Class distribution for 'at' relation per language:


at,FALSE,PROBABLE,TRUE
lang,,,
de,703,333,188
en,239,159,98
fr,2727,1087,636




Class distribution for 'isAt' relation per language:


isAt,FALSE,TRUE
lang,,
de,1090,134
en,419,77
fr,3984,466


## 6. Eval (Dev) Set Distribution per Language
Here we analyze the distribution of the labels in the evaluation sets for each language.

In [4]:
import json
import pandas as pd

# Set paths for dev files
dev_files = ["en-dev-cleaned.jsonl", "de-dev-cleaned.jsonl", "fr-dev-cleaned.jsonl"]

dev_pairs = []
for file_name in dev_files:
    file_path = base_path + file_name
    lang = file_name.split('-')[0]
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                record = json.loads(line)

                if 'sampled_pairs' in record:
                    for pair in record['sampled_pairs']:
                        pair_data = pair.copy()
                        pair_data['lang'] = lang
                        dev_pairs.append(pair_data)
                elif 'pairs' in record:
                    for pair in record['pairs']:
                        pair_data = pair.copy()
                        pair_data['lang'] = lang
                        dev_pairs.append(pair_data)
    except FileNotFoundError:
        print(f"File not found: {file_path}")

df_dev = pd.DataFrame(dev_pairs)

if not df_dev.empty:
    print(f"Total entity pairs loaded across all dev languages: {len(df_dev)}\n")

    # Calculate distribution for 'at' relation per language
    print("Class distribution for 'at' relation in dev set per language:")
    display(df_dev.groupby('lang')['at'].value_counts().unstack().fillna(0).astype(int))

    print("\n" + "="*50 + "\n")

    # Calculate distribution for 'isAt' relation per language
    print("Class distribution for 'isAt' relation in dev set per language:")
    display(df_dev.groupby('lang')['isAt'].value_counts().unstack().fillna(0).astype(int))
else:
    print("No dev data loaded. Please check the file paths.")

Total entity pairs loaded across all dev languages: 2081

Class distribution for 'at' relation in dev set per language:


at,FALSE,PROBABLE,TRUE
lang,,,
de,244,147,41
en,68,54,29
fr,952,367,179




Class distribution for 'isAt' relation in dev set per language:


isAt,FALSE,TRUE
lang,,
de,403,29
en,133,18
fr,1371,127
